## 1. Data Collection

In [30]:
import pandas as pd

train_df = pd.read_csv('../data/fraudTrain.csv')
test_df = pd.read_csv('../data/fraudTest.csv')
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


### About the Dataset (23 columns)

1. `Unnamed: 0` — copy of index
2. `trans_date_trans_time` — transaction date and time
3. `cc_num` — card number
4. `merchant` — merchant name
5. `category` — transaction category
6. `amt` — transaction amount ($)
7. `first` — customer name
8. `last` — customer last name
9. `gender` — customer gender
10. `street` — customer street address
11. `city` — customer city
12. `state` — customer state
13. `zip` — customer zip code
14. `lat` — customer latitude
15. `long` — customer longitude
16. `city_pop` — city population
17. `job` — customer occupation
18. `dob` — customer date of birth
19. `trans_num` — transaction ID
20. `unix_time` — transaction time (unix format)
21. `merch_lat` — merchant latitude
22. `merch_long` — merchant longitude
23. `is_fraud` — target variable (0=normal, 1=fraud)

In [31]:
train_df.shape

(1296675, 23)

In [32]:
train_df.columns.tolist()

['Unnamed: 0',
 'trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud']

In [33]:
train_df = pd.read_csv('../data/fraudTrain.csv')
test_df = pd.read_csv('../data/fraudTest.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df['is_fraud'].value_counts()

Train shape: (1296675, 23)
Test shape: (555719, 23)


is_fraud
0    1289169
1       7506
Name: count, dtype: int64

## 2. Exploratory Data Analysis / EDA 

In [35]:
train_df.groupby('is_fraud')['amt'].mean()

is_fraud
0     67.667110
1    531.320092
Name: amt, dtype: float64

This is a very significant signal! Fraudulent transactions are approximately 8 times higher in value than normal transactions. This suggests that fraudsters tend to try to make high-value purchases quickly once they gain access to a card, rather than making small "test" transactions (they want to maximize their "profit" before the card is blocked). Therefore, the amt column appears to be one of the model's strongest features.

Let's look at which categories fraudulent transactions are concentrated in for example, is there more fraud in online shopping or in grocery shopping?

This will show the percentage of transactions in each category that are fraudulent, sorted from highest to lowest.

In [36]:
print(train_df.groupby('category')['is_fraud'].mean().sort_values(ascending=False))

category
shopping_net      0.017561
misc_net          0.014458
grocery_pos       0.014098
shopping_pos      0.007225
gas_transport     0.004694
misc_pos          0.003139
grocery_net       0.002948
travel            0.002864
entertainment     0.002478
personal_care     0.002424
kids_pets         0.002114
food_dining       0.001651
home              0.001608
health_fitness    0.001549
Name: is_fraud, dtype: float64


Categories with the highest fraud rates:

shopping_net (online shopping): 1.76%
misc_net (other online): 1.45%
grocery_pos (grocery, physical POS): 1.41%
The lowest:

health_fitness: 0.15%
home: 0.16%
food_dining: 0.17%

Let's look at another pattern: Time-based
Let's also look at it by time, because fraud is generally concentrated during nighttime hours. First, we need to determine the time:

In [48]:
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])
train_df['hour'] = train_df['trans_date_trans_time'].dt.hour

test_df['trans_date_trans_time'] = pd.to_datetime(test_df['trans_date_trans_time'])
test_df['hour'] = test_df['trans_date_trans_time'].dt.hour


print(train_df.groupby('hour')['is_fraud'].mean().sort_values(ascending=False))

hour
22    0.028829
23    0.028374
1     0.015349
0     0.014940
2     0.014652
3     0.014239
5     0.001423
7     0.001327
14    0.001325
19    0.001236
18    0.001226
13    0.001225
15    0.001208
17    0.001192
16    0.001156
8     0.001153
21    0.001129
9     0.001114
4     0.001099
12    0.001027
11    0.000998
20    0.000952
10    0.000946
6     0.000946
Name: is_fraud, dtype: float64


This code is actually both an exploratory step and a small feature engineering step.

So what does this mean? This perfectly reflects real-world fraud behavior: fraudsters typically conduct transactions between midnight and early morning (10 PM - 3 AM). The logic is simple! These are the hours when the cardholder is asleep and unable to notice the transaction and intervene immediately. By the time the bank/user wakes up and notices, hours have passed.

## 2. Preprocessing / Data Manipulation

Let's start by dropping unnessasary columns

columns_to_drop = ['Unnamed: 0', 'first', 'last', 'street', 'cc_num', 'trans_num']

train_df = train_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

train_df.shape

In [51]:
columns_to_drop = ['Unnamed: 0', 'first', 'last', 'street', 'cc_num', 'trans_num']

train_df = train_df.drop(columns=columns_to_drop, errors='ignore')
test_df = test_df.drop(columns=columns_to_drop, errors='ignore')

train_df.shape

(1296675, 18)

In [56]:
test_df.shape

(555719, 19)

### Dropped Columns

The following columns were removed because they don't carry useful signal for fraud detection:

- `Unnamed: 0` — duplicate of the row index, no information
- `first` — customer first name, a random identifier unrelated to fraud
- `last` — customer last name, same reason
- `street` — customer's street address
- `cc_num` — card number, an identifier rather than a behavioral feature
- `trans_num` — unique per transaction, carries no repeatable pattern

**Result:** 23 columns → 17 columns (after also adding `hour`, currently at 18)

Let's calculate age from dob. In a new code cell:


In [55]:
train_df['dob'] = pd.to_datetime(train_df['dob'])
test_df['dob'] = pd.to_datetime(test_df['dob'])

train_df['age'] = (train_df['trans_date_trans_time'] - train_df['dob']).dt.days // 365
test_df['age'] = (test_df['trans_date_trans_time'] - test_df['dob']).dt.days // 365

print(train_df[['dob', 'trans_date_trans_time', 'age']].head())

         dob trans_date_trans_time  age
0 1988-03-09   2019-01-01 00:00:18   30
1 1978-06-21   2019-01-01 00:00:44   40
2 1962-01-19   2019-01-01 00:00:51   56
3 1967-01-12   2019-01-01 00:01:16   52
4 1986-03-28   2019-01-01 00:03:06   32


- Converts dob to a proper datetime type (same as we did for trans_date_trans_time)
- Calculates age by subtracting birth date from transaction date, getting the difference in days, then dividing by 365 to approximate years
- // is integer division — it rounds down, so someone who's "34.8 years old" becomes 34, matching how age normally works (you don't turn 35 until your birthday actually passes)
- Applied to both train and test, as always

In [58]:
print(train_df['age'].min())
print(train_df['age'].max())

13
95


Both are believable, though 13 is worth a small pause.

- Max = 95: Perfectly plausible. Someone born in the 1920s-30s still transacting in 2019-2020. No issue there.
- Min = 13: This is technically unusual. Most credit card issuers require cardholders to be 18+ (in the US, minors can be authorized users on a parent's account, but the account itself, and cc_num, would typically belong to an adult). A few possibilities:

Legitimate but unusual: some card products do allow authorized minor users, especially teens carrying a card linked to a parent's account
- Data quality artifact: since this is a simulated dataset (Sparkov generated, not real transactions), the generator may not have enforced a minimum age when creating synthetic customer profiles
- Not actually a data error, just an edge case worth noting but not necessarily something to fix

Let's see how common this is before deciding anything:

In [62]:
print(train_df[train_df['age'] < 18]['age'].value_counts().sort_index())

age
13      69
14    3855
15    6108
16    3125
17    2351
Name: count, dtype: int64


**Data quality note:** Age ranges from 13 to 95. Ages under 18 account for about 1.2% of transactions (69 to 6,108 rows per age from 13 to 17). These are kept as is since they likely represent authorized users on a parent's account, and the dates parse correctly with no signs of corruption.

In [66]:
print(train_df.groupby('is_fraud')['age'].agg(['mean', 'std', 'min', 'max']))

               mean        std  min  max
is_fraud                                
0         45.511960  17.398818   13   95
1         48.321609  18.864543   14   93


Age is a real but weak signal here, consistent with what we said earlier. It's not going to be a headline feature like amt or hour, but it's not noise either, there is a small, consistent shift toward older ages in the fraud group.

In [68]:
train_df.head()

,trans_date_trans_time,merchant,category,amt,gender,city,state,zip,lat,long,city_pop,job,dob,unix_time,merch_lat,merch_long,is_fraud,hour,age
0,2019-01-01 00:00:18,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,1325376018,36.011293,-82.048315,0,0,30
1,2019-01-01 00:00:44,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1325376044,49.159047,-118.186462,0,0,40
2,2019-01-01 00:00:51,fraud_Lind-Buckridge,entertainment,220.11,M,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,1325376051,43.150704,-112.154481,0,0,56
3,2019-01-01 00:01:16,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,1325376076,47.034331,-112.561071,0,0,52
4,2019-01-01 00:03:06,fraud_Keeling-Crist,misc_pos,41.96,M,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,1325376186,38.674999,-78.632459,0,0,32


Two common encoding approaches
1. Label Encoding: assigns each category a number (e.g. M=0, F=1). Simple, but implies an order that doesn't really exist (is "category 5" greater than "category 2"? not meaningfully). Works fine for tree-based models like XGBoost though, since trees split on thresholds rather than assuming linear order.
2. One-Hot Encoding: creates a separate 0/1 column for each category (e.g. category_travel, category_grocery_pos, etc). No false ordering implied, but can explode your column count if a feature has many unique values (like merchant or job, which could have hundreds).

## What we can do for this dataset

gender: simple label encoding (M/F, only 2 values, no ordering issue)
category: one-hot encoding (manageable number of categories, ~14)
merchant, city, job: these have too many unique values for one-hot (would create hundreds of columns). Simplest approach for a first model: drop them for now, and treat this as a limitation we can revisit later with more advanced encoding techniques

Let's check how many unique values merchant, city, and job actually have, to confirm they're too many:

In [69]:
print(train_df['merchant'].nunique())
print(train_df['city'].nunique())
print(train_df['job'].nunique())
print(train_df['state'].nunique())

693
894
494
51


Confirms it. merchant (693), city (894), and job (494) all have far too many unique values to one-hot encode without exploding the dataset into thousands of columns, that would slow training dramatically and likely hurt the model more than help it. state at 51 is borderline (matches the 50 US states plus DC), still a lot but more manageable if we wanted it later.
Plan for this step
Drop (too many unique values, not worth the complexity for a first model):

merchant
city
job
state (borderline, but let's keep things simple for now)

Encode:

gender: label encode (M/F to 0/1)
category: one-hot encode (~14 categories, manageable)

Let's do the drop first, in a new cell:

In [70]:
columns_to_drop_2 = ['merchant', 'city', 'job', 'state']

train_df = train_df.drop(columns=columns_to_drop_2, errors='ignore')
test_df = test_df.drop(columns=columns_to_drop_2, errors='ignore')

train_df.shape

(1296675, 15)

In [72]:
train_df['gender'] = train_df['gender'].map({'M': 0, 'F': 1})
test_df['gender'] = test_df['gender'].map({'M': 0, 'F': 1})

train_df['gender'].value_counts()

gender
1    709863
0    586812
Name: count, dtype: int64

This replaces every "M" with 0 and every "F" with 1, directly. Simple mapping, no library needed for just two values.